# Evaluation: were the Cardinals good at sending runners?

In this notebook I use the 2026 predictions to answer three questions:

1. How good is my model?
2. When is sending a runner worth the risk?
3. How did STL's sends and holds in 2026 compare with the rest of the league?

## Setup

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score

TEAM = "STL"
SEASON = 2026

## Load the predictions

- `train_sends`: the 2022–2025 sends my model learned from
- `plays`: every 2026 opportunity
- `sends`: the 2026 plays where the runner was sent

In [28]:
scored = pd.read_csv("../scored.csv")
train_sends = scored[scored.sent & (scored.season < SEASON)]
plays = scored[scored.season == SEASON]
sends = plays[plays.sent]

## How good is my model?

I can only grade the model on 2026 sends, because those are the only plays where I know if the runner was safe. I care about two questions:

1. **Can it tell risky plays from safe ones?** I use **AUC** for this. Pick one runner who was safe and one who was out. How often did the model give the safe runner the higher chance? 0.5 is a coin flip and 1.0 is perfect.
2. **Are its chances right?** AUC only checks the order of the plays, not the chances themselves, so I check the chances in the next table.

In [ ]:
print("2026 sends:", len(sends))
print("2026 outs:", (sends.safe == 0).sum())
print("AUC:", round(roc_auc_score(sends.safe, sends.p_safe), 3))

## Do the predicted chances match what happened?

This answers the second question.

I sort the 2026 sends from riskiest to safest and split them into 5 equal groups. In each group, *expected outs* = number of sends × (1 − average P(safe)). If the model's chances are right, expected outs should be close to actual outs. This matters because I use the model's chances to judge decisions later.

In [30]:
groups = pd.qcut(sends.p_safe, 5)
calibration = sends.groupby(groups).agg(sends=("safe", "size"),
                                        predicted_safe=("p_safe", "mean"),
                                        actual_safe=("safe", "mean"))
calibration["expected outs"] = calibration.sends * (1 - calibration.predicted_safe)
calibration["actual outs"] = calibration.sends * (1 - calibration.actual_safe)
calibration.round(3)

,sends,predicted_safe,actual_safe,expected outs,actual outs
p_safe,,,,,
"(0.733, 0.956]",599,0.926,0.922,44.356,47.0
"(0.956, 0.974]",598,0.966,0.967,20.312,20.0
"(0.974, 0.983]",601,0.979,0.980,12.618,12.0
"(0.983, 0.99]",595,0.987,0.992,7.707,5.0
"(0.99, 0.999]",598,0.994,0.995,3.761,3.0


## Do held runners look like the runners the model learned from?

My model only learned from runners who were sent, but later I use it on runners who were held. So I check that held runners aren't wildly different.

For each input, I show the share of 2026 held runners outside the range of the middle 98% of training sends (from the 1st to the 99th percentile). Small shares mean the model isn't guessing far outside what it has seen.

In [31]:
held = plays[~plays.sent]
inputs = ["launch_speed", "launch_angle", "spray_angle", "runner_speed", "fielder_arm"]
low = train_sends[inputs].quantile(0.01)
high = train_sends[inputs].quantile(0.99)


def share_outside(group):
    outside = (group[inputs] < low) | (group[inputs] > high)
    shares = outside.mean()
    shares["any input"] = outside.any(axis=1).mean()
    return shares


pd.DataFrame({
    "all held runners": share_outside(held),
    f"{TEAM} held runners": share_outside(held[held.bat_team == TEAM]),
}).round(3)

,all held runners,STL held runners
launch_speed,0.026,0.048
launch_angle,0.037,0.000
spray_angle,0.003,0.000
runner_speed,0.016,0.000
fielder_arm,0.018,0.000
any input,0.091,0.048


## Expected runs for the rest of the inning

*Run expectancy* is the average number of runs a team scores from a given situation until the end of the inning. For every plate appearance in 2022–2026, I take the batting team's score at the end of the half-inning minus its score when the plate appearance started. Then I average by outs and runners on base.

In `bases`, a number means a runner is on that base and `_` means it's empty, so `1 _ 3` is runners on first and third. I leave out home half-innings in the 9th inning or later, because a walk-off ends them early.

In [32]:
games = pd.read_csv("../statcast_game_data.csv",
                    usecols=["game_pk", "inning", "inning_topbot", "outs_when_up",
                             "on_1b", "on_2b", "on_3b", "bat_score", "post_bat_score"])
games = games[~((games.inning >= 9) & (games.inning_topbot == "Bot"))]

half_inning = games.groupby(["game_pk", "inning", "inning_topbot"])
games["runs_rest_of_inning"] = half_inning.post_bat_score.transform("max") - games.bat_score


def on_base(column, label):
    return games[column].notna().map({True: label, False: "_"})


games["bases"] = on_base("on_1b", "1") + " " + on_base("on_2b", "2") + " " + on_base("on_3b", "3")
run_expectancy = games.groupby(["bases", "outs_when_up"]).runs_rest_of_inning.mean().unstack()
run_expectancy.round(3)

outs_when_up,0,1,2
bases,,,
1 2 3,2.404,1.603,0.774
1 2 _,1.516,0.930,0.458
1 _ 3,1.838,1.190,0.497
1 _ _,0.886,0.527,0.223
_ 2 3,1.999,1.395,0.547
_ 2 _,1.141,0.675,0.321
_ _ 3,1.391,0.952,0.354
_ _ _,0.498,0.266,0.102


## When is sending worth it?

After a single to the outfield with a runner on second, three things can happen. I value each one with run expectancy:

| Outcome | What it looks like | Value |
|---|---|---|
| **hold** | runner on third, batter on first, same outs | run expectancy of `1 _ 3` |
| **score** | one run in, batter on first, same outs | 1 + run expectancy of `1 _ _` |
| **out** | batter on first, one more out | run expectancy of `1 _ _` with one more out (0 if it was the third out) |

If p is the chance the runner is safe, sending is worth p × score + (1 − p) × out. The *break-even* chance is where sending and holding are worth exactly the same:

break-even = (hold − out) / (score − out)

If a runner's chance of scoring is above break-even, sending is the better choice.

I made two simplifications: I assume the batter ends on first in every case, and a runner already on third scores either way, so only the runner from second changes between the outcomes.

In [33]:
run_values = pd.DataFrame({
    "hold": run_expectancy.loc["1 _ 3"],
    "score": 1 + run_expectancy.loc["1 _ _"],
    "out": [run_expectancy.loc["1 _ _", 1], run_expectancy.loc["1 _ _", 2], 0],
})
run_values["break_even"] = (run_values.hold - run_values.out) / (run_values.score - run_values.out)
run_values.round(3)

,hold,score,out,break_even
outs_when_up,,,,
0,1.838,1.886,0.527,0.965
1,1.190,1.527,0.223,0.742
2,0.497,1.223,0.000,0.406


## Judge each decision

For every 2026 play, I compare the value of sending with the value of holding, using the model's P(safe):

- `missed`: the runner was held, but sending was worth more
- `over`: the runner was sent, but holding was worth more
- `runs_lost_missed` and `runs_lost_over`: how many expected runs the decision cost
- `outs_above_expected`: for sends, actual outs minus expected outs, where actual is 1 if out and 0 if safe, and expected is 1 − P(safe). Below zero means fewer outs than expected. I leave held runners blank.

My model can't see where the outfielders were standing, how good a jump the runner got, or whether the coach waved him. So when I call a play `missed`, I mean similar sends usually scored, not that the coach was definitely wrong.

In [ ]:
def judge(plays):
    plays = plays.copy()
    p = plays.p_safe
    outs = plays.outs_when_up
    send_value = p * outs.map(run_values.score) + (1 - p) * outs.map(run_values.out)
    gain = send_value - outs.map(run_values.hold)

    plays["missed"] = ~plays.sent & (gain > 0)
    plays["over"] = plays.sent & (gain < 0)
    plays["runs_lost_missed"] = gain.where(plays.missed, 0)
    plays["runs_lost_over"] = -gain.where(plays.over, 0)
    plays["outs_above_expected"] = (1 - plays.safe) - (1 - p)
    return plays

## STL vs the rest of the league

`summarize` boils a group of plays down to five numbers. For all five, lower is better.

STL only had 158 opportunities, so some of its numbers could be luck. To measure that, I *bootstrap*: I make 2,000 pretend seasons by drawing STL's plays at random with replacement, so the same play can be drawn more than once, and recompute the numbers each time. The middle 95% of those results, from `low` to `high`, is the range STL's number could reasonably have landed in.

- League's number below `low`: STL is **worse**.
- League's number above `high`: STL is **better**.
- League's number in between: **no clear difference**.

By "league" I mean the other 29 teams. One warning: if STL's count is 0, every pretend season is also 0, so the range is 0 to 0. That makes "better" automatic, so I don't treat it as real evidence.

In [ ]:
def summarize(group):
    return pd.Series({
        "outs above expected, per send": group.outs_above_expected.mean(),
        "missed sends, per opportunity": group.missed.mean(),
        "over-aggressive sends, per opportunity": group.over.mean(),
        "runs lost to missed sends, per opportunity": group.runs_lost_missed.mean(),
        "runs lost to over-aggressive sends, per opportunity": group.runs_lost_over.mean(),
    })


def verdict(row):
    if row.low > row.league:
        return "worse"
    if row.high < row.league:
        return "better"
    return "no clear difference"


def compare(plays):
    stl = plays[plays.bat_team == TEAM]
    league = plays[plays.bat_team != TEAM]
    pretend_seasons = pd.DataFrame([summarize(stl.sample(frac=1, replace=True, random_state=i))
                                    for i in range(2000)])

    table = pd.DataFrame({
        TEAM: summarize(stl),
        "low": pretend_seasons.quantile(0.025),
        "high": pretend_seasons.quantile(0.975),
        "league": summarize(league),
    })
    table["better/worse"] = table.apply(verdict, axis=1)
    display(table.round(4))

    counts = pd.Series({
        "opportunities": len(stl),
        "sends": stl.sent.sum(),
        "held": (~stl.sent).sum(),
        "actual outs": (stl.safe == 0).sum(),
        "expected outs": (1 - stl.p_safe[stl.sent]).sum(),
        "missed sends": stl.missed.sum(),
        "over-aggressive sends": stl.over.sum(),
        "runs lost to missed sends": stl.runs_lost_missed.sum(),
    })
    display(counts.round(1))

## Results

In [ ]:
plays = judge(plays)
compare(plays)

## Missed sends by outs

The break-even changes a lot with the number of outs, so I count STL's missed sends at 0, 1 and 2 outs and compare STL's rate with the league's.

In [ ]:
plays["held"] = ~plays.sent
stl = plays[plays.bat_team == TEAM]
league = plays[plays.bat_team != TEAM]

by_outs = stl.groupby("outs_when_up").agg(opportunities=("held", "size"), held=("held", "sum"), missed=("missed", "sum"))
by_outs["missed per opportunity"] = by_outs.missed / by_outs.opportunities
by_outs["league missed per opportunity"] = league.groupby("outs_when_up").missed.mean()
by_outs.round(3)

## Save STL's missed sends for video review

I write one row per missed send to `stl_missed_sends.csv`, so each play can be checked on video. `savant_game` links to the game on Baseball Savant, and `inning` and `at_bat_number` point to the play within the game.

In [39]:
missed = stl[stl.missed].sort_values(["game_date", "inning"])
missed["savant_game"] = "https://baseballsavant.mlb.com/gamefeed?gamePk=" + missed.game_pk.astype(str)
missed.to_csv("../stl_missed_sends.csv", index=False)

missed[["game_date", "inning", "outs_when_up", "bb_type", "launch_speed",
        "runner_speed", "fielder_arm", "p_safe", "runs_lost_missed", "des"]].round(3)

,game_date,inning,outs_when_up,bb_type,launch_speed,runner_speed,fielder_arm,p_safe,runs_lost_missed,des
16528,2026-03-31,3,0,line_drive,94.7,29.7,84.9,0.991,0.036,JJ Wetherholt singles on a line drive to cente...
16372,2026-04-07,1,1,line_drive,97.4,27.6,87.5,0.936,0.253,Alec Burleson singles on a line drive to right...
16259,2026-04-12,6,1,line_drive,112.2,26.1,92.7,0.951,0.273,Jordan Walker singles on a sharp line drive to...
17138,2026-04-22,7,2,line_drive,105.0,25.0,89.3,0.896,0.599,Nathan Church singles on a sharp line drive to...
17067,2026-04-25,2,2,ground_ball,98.4,25.5,82.1,0.983,0.706,JJ Wetherholt singles on a ground ball to left...
16772,2026-05-08,5,1,ground_ball,99.4,25.6,92.9,0.934,0.251,Nolan Gorman singles on a ground ball to right...
16641,2026-05-13,4,0,line_drive,71.3,28.5,90.5,0.989,0.032,Nathan Church singles on a line drive to cente...
17935,2026-05-16,1,1,ground_ball,84.2,26.1,84.1,0.979,0.309,Jordan Walker singles on a ground ball to left...
17869,2026-05-19,10,0,line_drive,99.5,29.7,87.2,0.987,0.030,JJ Wetherholt singles on a line drive to right...
17787,2026-05-23,4,1,line_drive,91.5,28.5,85.8,0.960,0.285,Victor Scott II singles on a line drive to rig...
